
# Dense Retrieval Evaluation on SciFact

This notebook implements and evaluates an exact dense retrieval pipeline using the **SciFact** dataset from the BEIR benchmark.

We use a pretrained Sentence Transformer to encode scientific documents and queries into dense vectors, retrieve relevant documents using cosine similarity, and evaluate retrieval effectiveness against ground-truth relevance judgments.

The pipeline consists of:

1. Loading and preprocessing the SciFact dataset.
2. Generating normalized document and query embeddings.
3. Performing exact dense retrieval.
4. Selecting the Top-K documents.
5. Evaluating Recall@10 and nDCG@10.
6. Saving evaluation results.

## 1. Loading the SciFact Dataset

The dataset consists of three components:

- **Corpus:** Scientific documents to be retrieved.
- **Queries:** Scientific claims used as retrieval queries.
- **Qrels:** Ground-truth relevance judgments connecting queries to relevant documents.

The corpus contains 5,183 documents, and the test relevance judgments cover 300 evaluation queries.

We load the corpus, queries, and qrels using Hugging Face Datasets, then inspect sample records to understand their structure.


In [3]:
from datasets import load_dataset

corpus = load_dataset(
    "BeIR/scifact",
    "corpus",
    split="corpus"
)

queries = load_dataset(
    "BeIR/scifact",
    "queries",
    split="queries"
)

qrels = load_dataset(
    "BeIR/scifact-qrels",
    split="test"
)

print("Corpus size:", len(corpus))
print("Queries size:", len(queries))
print("Qrels size:", len(qrels))

Corpus size: 5183
Queries size: 1109
Qrels size: 339


In [4]:
print("=== Corpus Example ===")
print(corpus[0])

print("\n=== Query Example ===")
print(queries[0])

print("\n=== Qrels Example ===")
print(qrels[0])

=== Corpus Example ===
{'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of th


## 2. Data Preprocessing

We preprocess the SciFact dataset to prepare it for embedding generation and retrieval evaluation.

### Document Processing

For each document, we:

1. Extract its unique document ID.
2. Combine its title and text into a single string.
3. Create a mapping from document ID to its position in the embedding matrix.

The `doc_id_to_idx` dictionary connects original document IDs to their corresponding embedding matrix indices.

### Query and Relevance Judgment Processing

We construct two dictionaries:

- `query_dict`: Maps each query ID to its query text.
- `qrels_dict`: Maps each evaluated query ID to a set of relevant document IDs.

We include only relevance judgments with positive relevance scores and convert IDs to strings to ensure consistent matching across the dataset.


In [5]:
# Extract document IDs
doc_ids = [str(doc["_id"]) for doc in corpus]

# Combine title and text
doc_texts = [
    doc["title"] + " " + doc["text"]
    for doc in corpus
]

# Map document ID to its position in the embedding matrix
doc_id_to_idx = {
    doc_id: idx
    for idx, doc_id in enumerate(doc_ids)
}

print("Number of documents:", len(doc_ids))
print("First document ID:", doc_ids[0])
print("First document index:", doc_id_to_idx[doc_ids[0]])
print("First document text:", doc_texts[0][:200])

Number of documents: 5183
First document ID: 4983
First document index: 0
First document text: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the deve


In [6]:
# Query ID -> Query text
query_dict = {
    str(query["_id"]): query["text"]
    for query in queries
}

# Query ID -> Relevant document IDs
qrels_dict = {}

for row in qrels:
    query_id = str(row["query-id"])
    doc_id = str(row["corpus-id"])
    relevance = row["score"]

    if relevance > 0:
        if query_id not in qrels_dict:
            qrels_dict[query_id] = set()

        qrels_dict[query_id].add(doc_id)

print("Number of queries:", len(query_dict))
print("Number of evaluated queries:", len(qrels_dict))

first_query_id = next(iter(qrels_dict))

print("Query ID:", first_query_id)
print("Query text:", query_dict[first_query_id])
print("Relevant document IDs:", qrels_dict[first_query_id])

Number of queries: 1109
Number of evaluated queries: 300
Query ID: 1
Query text: 0-dimensional biomaterials show inductive properties.
Relevant document IDs: {'31715818'}



## 3. Generating Document Embeddings

We use the pretrained `sentence-transformers/all-MiniLM-L6-v2` model to encode the SciFact corpus.

Each document is represented by a 384-dimensional dense vector.

We enable L2 normalization during encoding using `normalize_embeddings=True`.

For a vector $v$, L2 normalization is defined as:

$$
\hat{v} = \frac{v}{\|v\|_2}
$$

For normalized query and document embeddings, cosine similarity is equivalent to their dot product:

$$
\operatorname{cosine}(q,d) = \hat{q} \cdot \hat{d}
$$

The resulting document embedding matrix has shape:

$$
D \in \mathbb{R}^{5183 \times 384}
$$

Each row represents one document in the SciFact corpus.


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = model.encode(
    doc_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)
print("Data type:", document_embeddings.dtype)
print("First document norm:", np.linalg.norm(document_embeddings[0]))

Batches: 100%|██████████| 162/162 [03:17<00:00,  1.22s/it]

Embedding shape: (5183, 384)
Data type: float32
First document norm: 1.0000001



## 4. Saving and Loading Document Embeddings

Generating embeddings requires running the Sentence Transformer model over the entire corpus.

To avoid repeating this computation, we save the document embedding matrix as a NumPy binary file:

`data/scifact_embeddings.npy`

We use `np.save()` to store the embeddings and `np.load()` to reload them.

After loading, we use `np.array_equal()` to verify that the saved embeddings are identical to the original embeddings.

These embeddings will also be reused in the ANN benchmark notebook, ensuring that exact and approximate retrieval methods use the same document representations.


In [8]:
from pathlib import Path

# Find the project data folder
cwd = Path.cwd()

if cwd.name == "notebooks":
    data_dir = cwd.parent / "data"
else:
    data_dir = cwd / "data"

data_dir.mkdir(parents=True, exist_ok=True)

# Save document embeddings
embedding_path = data_dir / "scifact_embeddings.npy"

np.save(embedding_path, document_embeddings)

print("Saved to:", embedding_path)
print("File size (MB):", embedding_path.stat().st_size / (1024 ** 2))

Saved to: C:\Users\weiting\Desktop\dense_retrieval\data\scifact_embeddings.npy
File size (MB): 7.5924072265625


In [9]:
loaded_embeddings = np.load(embedding_path)

print("Loaded shape:", loaded_embeddings.shape)

assert np.array_equal(
    document_embeddings,
    loaded_embeddings
)

print("Embeddings saved and loaded successfully!")

Loaded shape: (5183, 384)
Embeddings saved and loaded successfully!



## 5. Preparing Evaluation Queries

We select queries that appear in both `query_dict` and `qrels_dict`.

This ensures that every evaluated query has corresponding ground-truth relevance judgments.

We then encode the selected queries using the same Sentence Transformer model as the document corpus.

Query embeddings are also L2-normalized.

The resulting query embedding matrix has shape:

$$
Q \in \mathbb{R}^{300 \times 384}
$$

Using the same model and normalization procedure allows queries and documents to be compared in the same embedding space.


In [10]:
# Select queries that have relevance judgments
eval_query_ids = [
    query_id
    for query_id in qrels_dict
    if query_id in query_dict
]

eval_query_texts = [
    query_dict[query_id]
    for query_id in eval_query_ids
]

print("Evaluation queries:", len(eval_query_ids))
print("First query ID:", eval_query_ids[0])
print("First query text:", eval_query_texts[0])
print("Relevant docs:", qrels_dict[eval_query_ids[0]])

Evaluation queries: 300
First query ID: 1
First query text: 0-dimensional biomaterials show inductive properties.
Relevant docs: {'31715818'}


In [11]:
query_embeddings = model.encode(
    eval_query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Query embedding shape:", query_embeddings.shape)
print("First query norm:", np.linalg.norm(query_embeddings[0]))

Batches: 100%|██████████| 10/10 [00:00<00:00, 11.17it/s]

Query embedding shape: (300, 384)
First query norm: 1.0



## 6. Exact Dense Retrieval

We perform exact dense retrieval by computing cosine similarity between every evaluation query and every document.

Since both query and document embeddings are L2-normalized, we can compute similarity scores using matrix multiplication:

$$
S = QD^T
$$

where:

- $Q \in \mathbb{R}^{M \times d}$ is the query embedding matrix.
- $D \in \mathbb{R}^{N \times d}$ is the document embedding matrix.
- $S \in \mathbb{R}^{M \times N}$ is the similarity matrix.

For SciFact, the similarity matrix has shape:

$$
S \in \mathbb{R}^{300 \times 5183}
$$

Each entry $S_{ij}$ represents the similarity between query $i$ and document $j$.

This is an exact search because every query is compared against every document, without approximate candidate selection.


In [12]:
# Compute cosine similarity for all query-document pairs
similarity_matrix = query_embeddings @ document_embeddings.T

print("Similarity matrix shape:", similarity_matrix.shape)
print("First query scores:", similarity_matrix[0][:5])

Similarity matrix shape: (300, 5183)
First query scores: [0.13684508 0.08210057 0.10933483 0.08162077 0.13464166]



## 7. Efficient Top-K Retrieval

After computing the similarity matrix, we retrieve the 10 highest-scoring documents for each query.

Instead of sorting all document scores, we use `np.argpartition()` to efficiently identify the Top-K candidates.

The retrieval process consists of:

1. Select the indices of the K largest similarity scores using `np.argpartition()`.
2. Extract the similarity scores of the selected candidates.
3. Sort these K candidates in descending order.
4. Map the resulting matrix indices back to their original SciFact document IDs.

For a query with $N$ candidate documents, full sorting requires:

$$
O(N \log N)
$$

NumPy's partition-based selection has an average time complexity of:

$$
O(N)
$$

Sorting the selected K candidates requires:

$$
O(K \log K)
$$

Therefore, the combined average time complexity is:

$$
O(N + K \log K)
$$

This is especially useful when $K$ is much smaller than $N$.

Finally, we inspect the retrieved documents and their similarity scores to verify the retrieval output.


In [13]:
k = 10

# Step 1: Find the indices of the K largest scores
top_k_unsorted = np.argpartition(
    similarity_matrix,
    kth=-k,
    axis=1
)[:, -k:]

# Step 2: Extract the scores of these K documents
top_k_scores = np.take_along_axis(
    similarity_matrix,
    top_k_unsorted,
    axis=1
)

# Step 3: Sort only these K documents
sorted_order = np.argsort(
    top_k_scores,
    axis=1
)[:, ::-1]

top_k_indices = np.take_along_axis(
    top_k_unsorted,
    sorted_order,
    axis=1
)

print("Top-K shape:", top_k_indices.shape)
print("First query Top-K indices:", top_k_indices[0])

Top-K shape: (300, 10)
First query Top-K indices: [4132  766  697 1838 4241 2723 5115  166 2981  320]


In [14]:
retrieval_results = {}

for query_idx, query_id in enumerate(eval_query_ids):

    top_indices = top_k_indices[query_idx]

    retrieved_doc_ids = [
        doc_ids[doc_idx]
        for doc_idx in top_indices
    ]

    retrieval_results[query_id] = retrieved_doc_ids

print("First query ID:", eval_query_ids[0])
print("Retrieved document IDs:", retrieval_results[eval_query_ids[0]])

First query ID: 1
Retrieved document IDs: ['29638116', '4346436', '3874000', '10786948', '31715818', '17388232', '86129154', '927561', '19855358', '1769799']


In [15]:
first_query_id = eval_query_ids[0]

print("Query:")
print(query_dict[first_query_id])

print("\nTop-5 Retrieved Documents:")

for rank, doc_id in enumerate(
    retrieval_results[first_query_id][:5],
    start=1
):
    doc_idx = doc_id_to_idx[doc_id]

    score = similarity_matrix[0, doc_idx]

    print(f"\nRank {rank}")
    print(f"Document ID: {doc_id}")
    print(f"Score: {score:.4f}")
    print(f"Text: {doc_texts[doc_idx][:200]}...")

Query:
0-dimensional biomaterials show inductive properties.

Top-5 Retrieved Documents:

Rank 1
Document ID: 29638116
Score: 0.3540
Text: Complex Tissue and Disease Modeling using hiPSCs. Defined genetic models based on human pluripotent stem cells have opened new avenues for understanding disease mechanisms and drug screening. Many of ...

Rank 2
Document ID: 4346436
Score: 0.3306
Text: Nonlinear Elasticity in Biological Gels Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological functi...

Rank 3
Document ID: 3874000
Score: 0.2922
Text: Tissue Mechanics Orchestrate Wnt-Dependent Human Embryonic Stem Cell Differentiation. Regenerative medicine is predicated on understanding the mechanisms regulating development and applying these cond...

Rank 4
Document ID: 10786948
Score: 0.2916
Text: An efficient nonviral method to generate integration-free human-induced pluripotent stem cells from c


## 8. Evaluating Retrieval with Recall@K

We evaluate retrieval effectiveness by comparing retrieved documents against the ground-truth relevance judgments.

**Recall@K** measures the proportion of relevant documents successfully retrieved within the Top-K results.

For a query $q$, Recall@K is defined as:

$$
\operatorname{Recall@K}(q) =
\frac{|R_q \cap T_q^K|}{|R_q|}
$$

where:

- $R_q$ is the set of ground-truth relevant documents.
- $T_q^K$ is the set of retrieved documents within the Top-K results.

A Recall@10 value of 1.0 indicates that all ground-truth relevant documents were retrieved within the Top-10 results.

We calculate Recall@10 for each evaluation query and report the mean across all evaluated queries:

$$
\operatorname{MeanRecall@10} =
\frac{1}{M}
\sum_{i=1}^{M}
\operatorname{Recall@10}(q_i)
$$

In our experiment, the mean Recall@10 is approximately **0.7833**.


In [16]:
def recall_at_k(retrieved_docs, relevant_docs, k):
    top_k_docs = retrieved_docs[:k]

    hits = len(set(top_k_docs) & relevant_docs)

    return hits / len(relevant_docs)


recall_scores = []

for query_id in eval_query_ids:
    retrieved_docs = retrieval_results[query_id]
    relevant_docs = qrels_dict[query_id]

    score = recall_at_k(
        retrieved_docs,
        relevant_docs,
        k=10
    )

    recall_scores.append(score)


mean_recall = np.mean(recall_scores)

print(f"Mean Recall@10: {mean_recall:.4f}")

Mean Recall@10: 0.7833



## 9. Evaluating Ranking Quality with nDCG@K

Recall@K measures whether relevant documents are retrieved, but it does not consider their positions in the ranking.

We use **Normalized Discounted Cumulative Gain (nDCG)** to evaluate ranking quality.

### Discounted Cumulative Gain (DCG)

DCG assigns higher importance to relevant documents appearing near the top of the ranking.

For binary relevance:

$$
\operatorname{DCG@K} =
\sum_{i=1}^{K}
\frac{\operatorname{rel}_i}{\log_2(i+1)}
$$

where $\operatorname{rel}_i$ equals 1 if the document at rank $i$ is relevant, and 0 otherwise.

### Ideal Discounted Cumulative Gain (IDCG)

IDCG represents the maximum possible DCG for a query, obtained by placing relevant documents at the highest available ranks.

For binary relevance, the ideal ranking places relevant documents in the first $\min(|R_q|,K)$ positions.

### Normalized DCG (nDCG)

We normalize DCG using IDCG:

$$
\operatorname{nDCG@K} =
\frac{\operatorname{DCG@K}}{\operatorname{IDCG@K}}
$$

If IDCG is zero, our implementation returns 0.

nDCG ranges from 0 to 1, with higher values indicating better ranking quality.

We calculate nDCG@10 for every evaluation query and report the mean across all queries.

In our experiment, the mean nDCG@10 is approximately **0.6451**.


In [17]:
import math

def dcg_at_k(retrieved_docs, relevant_docs, k):
    dcg = 0.0

    for rank, doc_id in enumerate(retrieved_docs[:k], start=1):

        if doc_id in relevant_docs:
            dcg += 1 / math.log2(rank + 1)

    return dcg

In [18]:
def ndcg_at_k(retrieved_docs, relevant_docs, k):
    # Actual DCG
    dcg = dcg_at_k(retrieved_docs, relevant_docs, k)

    # Ideal DCG
    num_relevant = min(len(relevant_docs), k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, num_relevant + 1)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


ndcg_scores = []

for query_id in eval_query_ids:
    retrieved_docs = retrieval_results[query_id]
    relevant_docs = qrels_dict[query_id]

    score = ndcg_at_k(retrieved_docs, relevant_docs, k=10)
    ndcg_scores.append(score)

mean_ndcg = np.mean(ndcg_scores)

print(f"Mean nDCG@10: {mean_ndcg:.4f}")

Mean nDCG@10: 0.6451



## 10. Saving Evaluation Results

We save the exact dense retrieval evaluation results to a JSON file:

`results/exact_retrieval_metrics.json`

The output includes:

- Dataset and embedding model.
- Number of documents and evaluation queries.
- Mean Recall@10.
- Mean nDCG@10.

### Experimental Results

| Metric | Result |
|---|---:|
| Dataset | SciFact |
| Documents | 5,183 |
| Evaluation Queries | 300 |
| Embedding Dimension | 384 |
| Recall@10 | 0.7833 |
| nDCG@10 | 0.6451 |

These metrics provide a retrieval-quality baseline for our dense retrieval system.

In the next notebook, we compare exact search using FAISS IndexFlatIP with approximate search using HNSW, focusing on ANN Recall@10, search latency, and index size.


In [20]:
import json
from pathlib import Path

# Locate the project results directory
cwd = Path.cwd()

if cwd.name == "notebooks":
    results_dir = cwd.parent / "results"
else:
    results_dir = cwd / "results"

results_dir.mkdir(parents=True, exist_ok=True)
metrics = {
    "dataset": "SciFact",
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "num_documents": len(doc_ids),
    "num_queries": len(eval_query_ids),
    "recall@10": float(mean_recall),
    "ndcg@10": float(mean_ndcg)
}

output_path = results_dir / "exact_retrieval_metrics.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print("Saved to:", output_path)
print(json.dumps(metrics, indent=4))

Saved to: C:\Users\weiting\Desktop\dense_retrieval\results\exact_retrieval_metrics.json
{
    "dataset": "SciFact",
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "num_documents": 5183,
    "num_queries": 300,
    "recall@10": 0.7833333333333333,
    "ndcg@10": 0.6450816521455776
}
